# 📱 Perbandingan Algoritma Decision Tree dan K-Nearest Neighbors (KNN) untuk Klasifikasi Tingkat Kecanduan Smartphone pada Remaja

**Mata Kuliah:** Fundamental Data Analyst  
**Dataset:** `teen_phone_addiction_dataset.csv` (Dataset Asli) & `teen_phone_addiction_prepared.csv` (Dataset Hasil Olah)  
**Metodologi:** CRISP-DM (*Cross-Industry Standard Process for Data Mining*)

---

### 📌 1. Judul dan Tujuan Penelitian

#### Latar Belakang & Pertanyaan Penelitian
Penggunaan ponsel pintar (*smartphone*) di kalangan remaja saat ini mencakup berbagai aktivitas, mulai dari media sosial, permainan (*gaming*), penjelajahan web (*browsing*), hingga keperluan belajar (*education*). Meskipun smartphone memberikan kemudahan akses informasi dan komunikasi, intensitas pemakaian yang berlebihan berpotensi berasosiasi dengan penurunan kualitas tidur, performa akademik, serta kesejahteraan psikososial.

Penelitian ini bertujuan untuk menjawab pertanyaan penelitian berikut:
1. **Bagaimana karakteristik dan pola penggunaan smartphone pada remaja dalam dataset ini?**
2. **Fitur-fitur aktivitas apa saja yang memiliki keterkaitan (asosiasi) dengan skor tingkat kecanduan smartphone (`Addiction_Level`)?**
3. **Bagaimana alternatif pembagian batas kategori (*discretization*) tingkat kecanduan menjadi kelas Rendah, Sedang, dan Tinggi yang rasional dan transparan?**
4. **Bagaimana perbandingan performa algoritma *Decision Tree* dan *K-Nearest Neighbors* (KNN) dalam mengklasifikasikan tingkat kecanduan smartphone pada data uji berdasarkan metrik Accuracy, Macro Precision, Macro Recall, dan Macro F1-Score?**

> ⚠️ **BATASAN & ETIKA ANALISIS (DISCLAIMER PENTING):**  
> Notebook ini adalah latihan analisis data dan eksplorasi *machine learning* untuk kebutuhan akademik. Analisis ini **BUKAN alat diagnosis medis, klinis, atau psikologis**. Dataset tidak memuat definisi atau batas klinis baku tingkat kecanduan. Penentuan kategori di dalam notebook ini bersifat eksploratif dan matematis untuk tujuan klasifikasi data, bukan penegakan diagnosis psikologis pada individu.

---  
## 📦 2. Import Library dan Membaca Data

**Tujuan:**  
Memuat pustaka Python yang dibutuhkan untuk pengolahan data (`pandas`, `numpy`), visualisasi (`matplotlib`, `seaborn`), dan pemodelan *machine learning* (`scikit-learn`), serta membaca dataset dari file CSV asli `teen_phone_addiction_dataset.csv` dengan *path* relatif.

In [ ]:
# Import pustaka standar untuk manipulasi dan komputasi numerik
import os
import numpy as np
import pandas as pd

# Import pustaka visualisasi data
import matplotlib.pyplot as plt
import seaborn as sns

# Konfigurasi visualisasi agar informatif dan estetis
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
sns.set_palette('tab10')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Import pustaka scikit-learn untuk pra-pemrosesan dan pemodelan
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# Pengaturan seed agar hasil dapat direplikasi secara konsisten
RANDOM_STATE = 42
print("✅ Seluruh pustaka berhasil diimpor dengan sukses!")

In [ ]:
# Membaca dataset asli teen_phone_addiction_dataset.csv dengan path relatif
file_path = 'teen_phone_addiction_dataset.csv'

if not os.path.exists(file_path):
    raise FileNotFoundError(
        f"File '{file_path}' tidak ditemukan di folder kerja ({os.getcwd()}). "
        f"Pastikan file CSV berada satu folder dengan notebook ini."
    )

df = pd.read_csv(file_path)
print(f"✅ Dataset asli berhasil dibaca dari: '{file_path}'")
print(f"Dimensi data: {df.shape[0]:,} baris dan {df.shape[1]} kolom.")

In [ ]:
# Menampilkan 5 baris pertama data asli
print("--- 5 Baris Pertama Dataset ---")
display(df.head())

# Ringkasan informasi tipe data dan nilai non-null
print("\n--- Ringkasan Struktur Data (df.info()) ---")
df.info()

In [ ]:
# Pemeriksaan nilai kosong (missing values) dan baris duplikat
missing_counts = df.isnull().sum()
total_duplicates = df.duplicated().sum()

print("--- Pengecekan Missing Values ---")
if missing_counts.sum() == 0:
    print("✅ Tidak ditemukan missing values pada seluruh kolom.")
else:
    print(missing_counts[missing_counts > 0])

print(f"\n--- Pengecekan Baris Duplikat ---")
print(f"Jumlah baris duplikat identik: {total_duplicates} baris.")

### 📝 Interpretasi Mahasiswa (Tahap 2):
- **Ukuran Data:** Dataset memuat 3.000 baris observasi remaja dengan 25 atribut.
- **Kebersihan Awal:** Tidak ditemukan *missing values* maupun baris duplikat, sehingga data relatif bersih dari segi kelengkapan dasar.
- **Tipe Data:** Mayoritas atribut berupa numerik (`float64` dan `int64`), serta beberapa atribut bertipe teks/kategorikal (`object`) seperti `Name`, `Gender`, `Location`, `School_Grade`, dan `Phone_Usage_Purpose`.

---  
## 🔍 3. Data Understanding (Eksplorasi Data)

**Tujuan:**  
1. Memahami peran masing-masing atribut (variabel target awal, fitur prediktor, dan kolom identitas).
2. Memeriksa rentang usia dan statistik ringkas target `Addiction_Level`.
3. Memvisualisasikan distribusi skor dan kaitannya dengan fitur durasi penggunaan (`Daily_Usage_Hours`, `Time_on_Social_Media`, `Time_on_Gaming`, `Time_on_Education`, `Phone_Checks_Per_Day`).
4. Melakukan interpretasi deskriptif secara hati-hati tanpa membuat klaim kausalitas (sebab-akibat).

In [ ]:
# Pengelompokan peran kolom berdasarkan fungsi analisis
col_target = 'Addiction_Level'
col_identifiers = ['ID', 'Name', 'Location']
col_demografi = ['Age', 'Gender', 'School_Grade']
col_perilaku = [
    'Daily_Usage_Hours', 'Screen_Time_Before_Bed', 'Phone_Checks_Per_Day',
    'Apps_Used_Daily', 'Time_on_Social_Media', 'Time_on_Gaming',
    'Time_on_Education', 'Phone_Usage_Purpose', 'Weekend_Usage_Hours'
]
col_psikososial = [
    'Sleep_Hours', 'Academic_Performance', 'Social_Interactions', 'Exercise_Hours',
    'Anxiety_Level', 'Depression_Level', 'Self_Esteem', 'Parental_Control', 'Family_Communication'
]

print("Peran Variabel dalam Dataset:")
print(f"• Target Awal Kontinu   : {col_target}")
print(f"• Identitas Pribadi     : {col_identifiers}")
print(f"• Atribut Demografis    : {col_demografi}")
print(f"• Perilaku Penggunaan   : {col_perilaku}")
print(f"• Dampak & Psikososial  : {col_psikososial}")

In [ ]:
# Analisis rentang usia dan tingkat sekolah responden
print("--- Statistik Deskriptif Rentang Usia (Age) ---")
print(df['Age'].describe())

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
sns.countplot(data=df, x='Age', hue='Age', palette='crest', legend=False, ax=ax[0])
ax[0].set_title('Distribusi Usia Responden (Tahun)')
ax[0].set_xlabel('Usia')
ax[0].set_ylabel('Jumlah Responden')

grade_order = ['7th', '8th', '9th', '10th', '11th', '12th']
sns.countplot(data=df, x='School_Grade', order=grade_order, hue='School_Grade', palette='viridis', legend=False, ax=ax[1])
ax[1].set_title('Distribusi Jenjang Kelas Sekolah')
ax[1].set_xlabel('Kelas')
ax[1].set_ylabel('Jumlah Responden')
plt.tight_layout()
plt.show()

In [ ]:
# Ringkasan statistik dan visualisasi distribusi target Addiction_Level
target_stats = df['Addiction_Level'].describe()
target_skew = df['Addiction_Level'].skew()
target_median = df['Addiction_Level'].median()

print("--- Ringkasan Statistik Skor Addiction_Level ---")
print(target_stats)
print(f"Median   : {target_median:.2f}")
print(f"Skewness : {target_skew:.3f} (kemencengan distribusi)")

fig, (ax_box, ax_hist) = plt.subplots(
    2, 1, figsize=(10, 7), sharex=True,
    gridspec_kw={'height_ratios': [0.25, 0.75]}
)

# Boxplot horizontal
sns.boxplot(x=df['Addiction_Level'], color='skyblue', ax=ax_box)
ax_box.set(xlabel='')
ax_box.set_title('Distribusi Skor Addiction_Level (Boxplot & Histogram)', fontsize=14)

# Histogram dengan KDE
sns.histplot(df['Addiction_Level'], kde=True, color='teal', bins=30, ax=ax_hist)
ax_hist.axvline(target_stats['mean'], color='red', linestyle='--', label=f"Mean ({target_stats['mean']:.2f})")
ax_hist.axvline(target_median, color='gold', linestyle='-', linewidth=2, label=f"Median ({target_median:.2f})")
ax_hist.set_xlabel('Skor Addiction_Level')
ax_hist.set_ylabel('Frekuensi')
ax_hist.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Visualisasi hubungan Addiction_Level dengan fitur penggunaan smartphone
fitur_penggunaan = [
    'Daily_Usage_Hours',
    'Time_on_Social_Media',
    'Time_on_Gaming',
    'Time_on_Education',
    'Phone_Checks_Per_Day'
]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(fitur_penggunaan):
    # Menggunakan scatterplot dengan garis regresi tren linear
    sns.regplot(
        data=df, x=col, y='Addiction_Level',
        scatter_kws={'alpha': 0.25, 'color': 'steelblue'},
        line_kws={'color': 'crimson', 'linewidth': 2},
        ax=axes[i]
    )
    corr_val = df[col].corr(df['Addiction_Level'])
    axes[i].set_title(f"{col} vs Addiction_Level\n(Korelasi Pearson: r = {corr_val:+.2f})")
    axes[i].set_ylabel('Addiction Level')

# Subplot ke-6 untuk korelasi dengan Weekend_Usage_Hours sebagai pelengkap
sns.regplot(
    data=df, x='Weekend_Usage_Hours', y='Addiction_Level',
    scatter_kws={'alpha': 0.25, 'color': 'darkgreen'},
    line_kws={'color': 'crimson', 'linewidth': 2},
    ax=axes[5]
)
corr_weekend = df['Weekend_Usage_Hours'].corr(df['Addiction_Level'])
axes[5].set_title(f"Weekend_Usage_Hours vs Addiction_Level\n(Korelasi Pearson: r = {corr_weekend:+.2f})")
axes[5].set_ylabel('Addiction Level')

plt.suptitle('Hubungan Antara Fitur Perilaku Smartphone dan Skor Addiction_Level', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Analisis Korelasi terhadap Addiction_Level
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr_matrix = df[numeric_cols].corr()
corr_with_target = corr_matrix['Addiction_Level'].sort_values(ascending=False)

plt.figure(figsize=(10, 8))
corr_with_target.drop('Addiction_Level').plot(kind='barh', color='teal')
plt.axvline(0, color='black', linestyle='--', linewidth=0.8)
plt.title('Korelasi Linear Seluruh Fitur Numerik terhadap Addiction_Level')
plt.xlabel('Koefisien Korelasi Pearson (r)')
plt.ylabel('Fitur Numerik')
plt.tight_layout()
plt.show()

print("Top 5 Fitur dengan Korelasi Positif Tertinggi terhadap Addiction_Level:")
print(corr_with_target.drop('Addiction_Level').head(5))
print("\nTop 5 Fitur dengan Korelasi Negatif Tertinggi terhadap Addiction_Level:")
print(corr_with_target.drop('Addiction_Level').tail(5))

### 📝 Interpretasi Hasil Eksplorasi (Tahap 3):
- **Rentang Responden:** Dataset mencakup remaja berusia antara 13 hingga 19 tahun pada jenjang kelas 7 hingga 12.
- **Distribusi Target:** Skor `Addiction_Level` memiliki sebaran bernilai kontinu (misal skala hingga 10). Sebagian besar observasi terkonsentrasi di nilai menengah hingga tinggi.
- **Pola Keterkaitan (Bukan Kausalitas):**
  - Durasi penggunaan harian (`Daily_Usage_Hours`) dan durasi media sosial (`Time_on_Social_Media`) memperlihatkan kecenderungan korelasi positif terhadap skor kecanduan.
  - Sebaliknya, fitur seperti waktu penggunaan untuk edukasi (`Time_on_Education`) atau durasi tidur (`Sleep_Hours`) memperlihatkan hubungan yang berbeda atau korelasi negatif/lemah.
  - *Catatan Metodologis:* Hubungan korelasi ini menunjukkan adanya **asosiasi statistik**, bukan bukti bahwa penggunaan aplikasi tertentu secara langsung 'menyebabkan' kecanduan.

---  
## 🏷️ 4. Pengecekan Target dan Opsi Kategorisasi

**Tujuan:**  
1. Menjelaskan bahwa atribut `Addiction_Level` adalah angka desimal kontinu, bukan label kategori.
2. Menampilkan beberapa opsi pemotongan skor (*discretization*) menjadi 3 kelas: **Rendah**, **Sedang**, dan **Tinggi**.
3. Memeriksa jumlah dan proporsi data pada masing-masing opsi serta menangani nilai batas yang identik secara aman.
4. Menyediakan konfigurasi `BATAS_KATEGORI` yang dapat langsung dijalankan (default kuantil seimbang) atau disesuaikan secara manual.

In [ ]:
# Fungsi pembantu untuk mensimulasikan pemotongan skor dan mengevaluasi keseimbangan kelas
def evaluasi_opsi_kategori(nama_opsi, batas_bawah_sedang, batas_bawah_tinggi, scores):
    """
    Mengevaluasi pembagian 3 kelas berdasarkan 2 nilai ambang batas:
    - Rendah : score < batas_bawah_sedang
    - Sedang : batas_bawah_sedang <= score < batas_bawah_tinggi
    - Tinggi : score >= batas_bawah_tinggi
    """
    kondisi = [
        (scores < batas_bawah_sedang),
        (scores >= batas_bawah_sedang) & (scores < batas_bawah_tinggi),
        (scores >= batas_bawah_tinggi)
    ]
    pilihan = ['Rendah', 'Sedang', 'Tinggi']
    kategori = np.select(kondisi, pilihan, default='Tidak Terdefinisi')
    
    df_kat = pd.Series(kategori).value_counts().reindex(pilihan, fill_value=0)
    df_pct = (df_kat / len(scores)) * 100
    
    hasil = pd.DataFrame({
        'Jumlah Data': df_kat,
        'Persentase (%)': df_pct.round(2)
    })
    
    print(f"\n=== {nama_opsi} ===")
    print(f"Ambang Batas : [Rendah < {batas_bawah_sedang:.2f}] | [{batas_bawah_sedang:.2f} <= Sedang < {batas_bawah_tinggi:.2f}] | [Tinggi >= {batas_bawah_tinggi:.2f}]")
    display(hasil)
    
    # Pengecekan keamanan jika ada kelas yang kosong atau sangat tidak seimbang
    if (df_kat == 0).any():
        print("⚠️ PERINGATAN: Terdapat kelas yang memiliki 0 sampel data!")
    min_pct = df_pct.min()
    if min_pct < 10.0:
        print(f"⚠️ PERINGATAN KETIDAKSEIMBANGAN: Kelas terkecil hanya memiliki {min_pct:.1f}% data.")
        
    return batas_bawah_sedang, batas_bawah_tinggi

In [ ]:
scores = df['Addiction_Level']
min_s, max_s = scores.min(), scores.max()

# -------------------------------------------------------------------------
# OPSI A: Kuantil / Tertil (Persentil 33.33% dan 66.67%)
# -------------------------------------------------------------------------
q1 = np.percentile(scores, 100/3)
q2 = np.percentile(scores, 200/3)
# Penanganan aman jika nilai kuantil bernilai identik
if q1 == q2:
    q2 = q1 + 0.01
evaluasi_opsi_kategori("Opsi A: Berbasis Kuantil (Tertil 1/3 & 2/3)", q1, q2, scores)

# -------------------------------------------------------------------------
# OPSI B: Equal-Width Interval (Rentang Nilai Min-Max Dibagi 3 Sama Lebar)
# -------------------------------------------------------------------------
lebar = (max_s - min_s) / 3
b_sedang_eq = min_s + lebar
b_tinggi_eq = min_s + (2 * lebar)
evaluasi_opsi_kategori("Opsi B: Equal-Width Interval (Rentang Sama Lebar)", b_sedang_eq, b_tinggi_eq, scores)

# -------------------------------------------------------------------------
# OPSI C: Ambang Batas Teoretis / Heuristik Skala (Contoh: 6.0 dan 8.5)
# -------------------------------------------------------------------------
evaluasi_opsi_kategori("Opsi C: Ambang Batas Heuristik Skala [6.0, 8.5]", 6.0, 8.5, scores)

print("\n💡 CATATAN DISKUSI KELOMPOK:")
print("• Opsi Kuantil (A) menjaga agar data terbagi merata (~33.3% per kelas), mencegah bias kelas mayoritas pada model.")
print("• Opsi Equal-Width (B) dan Heuristik (C) mengikuti nilai numerik murni, namun berisiko menghasilkan ketidakseimbangan kelas (*class imbalance*).")
print("• Ingat bahwa pembagian ini HANYA eksplorasi analitik, BUKAN diagnosis klinis!")

In [ ]:
# ==============================================================================
# ⚙️ CELL KONFIGURASI BATAS KATEGORI TINGKAT KECANDUAN (USER INPUT)
# ==============================================================================
# Format: [batas_bawah_sedang, batas_bawah_tinggi]
# Aturan:
#   - Kelas 'Rendah' : skor < batas_bawah_sedang
#   - Kelas 'Sedang' : batas_bawah_sedang <= skor < batas_bawah_tinggi
#   - Kelas 'Tinggi' : skor >= batas_bawah_tinggi
#
# Secara default, notebook ini menggunakan nilai kuantil seimbang agar langsung dapat
# dieksekusi tanpa error. Anda dapat mengganti nilai ini sesuai kesepakatan kelompok:
# Contoh manual: BATAS_KATEGORI = [6.0, 8.5]

BATAS_KATEGORI = [round(float(q1), 2), round(float(q2), 2)]

print(f"✅ Batas Kategori Aktif: {BATAS_KATEGORI}")
print(f"   - Rendah : skor < {BATAS_KATEGORI[0]}")
print(f"   - Sedang : {BATAS_KATEGORI[0]} <= skor < {BATAS_KATEGORI[1]}")
print(f"   - Tinggi : skor >= {BATAS_KATEGORI[1]}")

---  
## 💼 5. Business Understanding

**Tujuan:**  
Menghubungkan permasalahan bisnis/akademik dengan klasifikasi tingkat kecanduan smartphone serta menegaskan tujuan perbandingan model.

#### 1. Masalah Utama
Kecanduan smartphone pada remaja merupakan isu multidimensional yang berinteraksi dengan pola tidur, prestasi sekolah, dan tingkat kecemasan. Dalam konteks institusi pendidikan atau bimbingan konseling, memiliki sistem peringatan dini yang mampu **mengelompokkan tingkat risiko kecanduan** menjadi kelas Rendah, Sedang, dan Tinggi dapat membantu konselor memprioritaskan edukasi dan pendampingan.

#### 2. Tujuan Perbandingan Algoritma
Kami membandingkan dua paradigma algoritma klasifikasi yang berbeda:
1. **Decision Tree (Pohon Keputusan):**  
   - Bersifat *white-box* (mudah diinterpretasikan).
   - Menghasilkan aturan *if-then* hierarkis yang transparan sehingga mudah dipahami oleh guru, orang tua, atau pihak non-teknis.
2. **K-Nearest Neighbors (KNN):**  
   - Bersifat *instance-based learning* (non-parametrik).
   - Mengklasifikasikan remaja berdasarkan kedekatan/kemiripan profil dengan kasus serupa di ruang fitur multi-dimensi.

#### 3. Manfaat Nyata & Realistis
- Mengetahui algoritma mana yang memberikan generalisasi terbaik pada data uji.
- Memahami *trade-off* antara interpretabilitas (*Decision Tree*) dan akurasi berbasis kemiripan lokal (*KNN*).
- Memberikan wawasan berbasis bukti (*data-driven*) mengenai atribut-atribut kunci yang membedakan kelompok berisiko tinggi tanpa membuat klaim diagnosis klinis yang berlebihan.

---  
## 🛠️ 6. Persiapan Data (Data Preparation)

Fase ini sangat krusial dalam siklus CRISP-DM untuk memastikan kualitas data, mencegah kebocoran informasi (*data leakage*), dan mempersiapkan representasi matematis yang optimal bagi algoritma pemodelan:

1. **Kategorisasi Target:** Mengonversi `Addiction_Level` menjadi label diskrit `Addiction_Category` ('Rendah', 'Sedang', 'Tinggi').
2. **Pencegahan Data Leakage & Eliminasi Identifier:**
   - Mengeluarkan `Addiction_Level` dari prediktor $X$ (*target leakage*).
   - Mengeluarkan kolom identitas `ID`, `Name`, dan `Location` (*identity leakage & high cardinality noise*).
3. **Rekayasa Fitur (Feature Engineering Domain-Spesifik):**
   - `Screen_to_Sleep_Ratio`: Rasio durasi pemakaian layar harian terhadap jam tidur.
   - `Weekend_Weekday_Usage_Diff`: Selisih durasi pemakaian akhir pekan terhadap hari kerja.
   - `Productive_Usage_Ratio`: Porsi waktu yang dialokasikan untuk edukasi terhadap total penggunaan harian.
   - `Mental_Health_Score`: Indeks gabungan rata-rata antara tingkat kecemasan (`Anxiety_Level`) dan depresi (`Depression_Level`).
4. **Penyimpanan Dataset Siap Pakai:** Mengekspor hasil persiapan ke file `teen_phone_addiction_prepared.csv`.
5. **Stratified Train-Test Split (80:20):** Pembagian data latih dan uji secara proporsional.
6. **Penyandian Fitur & Penskalaan:** Menggunakan `ColumnTransformer` (`StandardScaler` untuk numerik, `OneHotEncoder` untuk kategorikal) yang di-fit secara ketat hanya pada data latih melalui `Pipeline`.

In [ ]:
# 1. Membuat kolom target kategori berdasarkan batas yang disetujui
b_sedang, b_tinggi = BATAS_KATEGORI[0], BATAS_KATEGORI[1]

conditions = [
    (df['Addiction_Level'] < b_sedang),
    (df['Addiction_Level'] >= b_sedang) & (df['Addiction_Level'] < b_tinggi),
    (df['Addiction_Level'] >= b_tinggi)
]
choices = ['Rendah', 'Sedang', 'Tinggi']
df['Addiction_Category'] = np.select(conditions, choices, default='Sedang')

print("Sebaran Kelas Target (Addiction_Category):")
counts = df['Addiction_Category'].value_counts().reindex(choices)
pcts = (counts / len(df)) * 100
display(pd.DataFrame({'Jumlah Responden': counts, 'Persentase (%)': pcts.round(2)}))

In [ ]:
# 2. Rekayasa Fitur (Feature Engineering)
print("⏳ Melakukan Feature Engineering domain-spesifik...")

# A. Rasio Waktu Layar terhadap Durasi Tidur
df['Screen_to_Sleep_Ratio'] = np.round(df['Daily_Usage_Hours'] / (df['Sleep_Hours'] + 1e-5), 2)

# B. Selisih Pemakaian Akhir Pekan vs Hari Kerja
df['Weekend_Weekday_Usage_Diff'] = np.round(df['Weekend_Usage_Hours'] - df['Daily_Usage_Hours'], 2)

# C. Rasio Penggunaan Edukatif terhadap Total Penggunaan
df['Productive_Usage_Ratio'] = np.round(df['Time_on_Education'] / (df['Daily_Usage_Hours'] + 1e-5), 2)

# D. Indeks Komposit Kesehatan Mental (Rata-rata Anxiety & Depression)
df['Mental_Health_Score'] = np.round((df['Anxiety_Level'] + df['Depression_Level']) / 2.0, 2)

# E. Ordinal Mapping untuk Jenjang Kelas Sekolah (7th: 7, ..., 12th: 12)
grade_mapping = {'7th': 7, '8th': 8, '9th': 9, '10th': 10, '11th': 11, '12th': 12}
df['School_Grade_Num'] = df['School_Grade'].map(grade_mapping).fillna(9).astype(int)

print("✅ Fitur baru berhasil dibuat: Screen_to_Sleep_Ratio, Weekend_Weekday_Usage_Diff, Productive_Usage_Ratio, Mental_Health_Score, School_Grade_Num.")

In [ ]:
# 3. Menyimpan Dataset Hasil Persiapan ke File CSV 'teen_phone_addiction_prepared.csv'
prepared_file_path = 'teen_phone_addiction_prepared.csv'
df.to_csv(prepared_file_path, index=False)

print(f"💾 Dataset hasil persiapan berhasil disimpan ke: '{prepared_file_path}'")
print(f"Dimensi dataset siap pakai: {df.shape[0]} baris dan {df.shape[1]} kolom.")
display(df[['ID', 'Daily_Usage_Hours', 'Screen_to_Sleep_Ratio', 'Mental_Health_Score', 'Addiction_Category']].head())

In [ ]:
# 4. Pemisahan Fitur (X) dan Target (y) serta Pembagian Data (Train-Test Split)
y = df['Addiction_Category']

# Eliminasi identifier dan target leakage
cols_to_drop = ['Addiction_Category', 'Addiction_Level', 'ID', 'Name', 'Location', 'School_Grade']
X = df.drop(columns=[col for col in cols_to_drop if col in df.columns])

# Pembagian stratified 80:20
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f"Jumlah fitur X           : {X.shape[1]} kolom")
print(f"Ukuran data latih (X_train): {X_train.shape[0]} sampel")
print(f"Ukuran data uji (X_test)   : {X_test.shape[0]} sampel")

In [ ]:
# 5. Setup ColumnTransformer untuk Preprocessing Pipeline
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Fitur Numerik ({len(numeric_features)})    : {numeric_features}")
print(f"Fitur Kategorikal ({len(categorical_features)})  : {categorical_features}")

# Preprocessor untuk KNN (Wajib StandardScaler untuk seluruh fitur numerik)
knn_preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

# Preprocessor untuk Decision Tree
dt_preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

---  
## 🤖 7. Pemodelan (Modeling) dan Hyperparameter Tuning

Pada tahap ini, kedua algoritma dilatih dan disetel parameternya menggunakan **Stratified 5-Fold Cross Validation** HANYA pada data latih (`X_train`):
- **Decision Tree Classifier:** Mencari kombinasi optimal kedalaman pohon (`max_depth`), batas pemisahan simpul (`min_samples_split`), dan kriteria kemurnian (`gini` vs `entropy`).
- **K-Nearest Neighbors (KNN):** Mencari jumlah tetangga terdekat $k$ (`n_neighbors`), skema pembobotan (`uniform` vs `distance`), dan metrik jarak (`euclidean` vs `manhattan`).

In [ ]:
# Pemodelan & Tuning: Decision Tree
dt_pipeline = Pipeline([
    ('prep', dt_preprocessor),
    ('clf', DecisionTreeClassifier(random_state=RANDOM_STATE))
])

dt_param_grid = {
    'clf__max_depth': [3, 4, 5, 7, 10, None],
    'clf__min_samples_split': [2, 5, 10],
    'clf__min_samples_leaf': [1, 2, 4],
    'clf__criterion': ['gini', 'entropy']
}

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print("⏳ Melatih dan melakukan tuning Decision Tree dengan 5-Fold CV...")
dt_grid = GridSearchCV(
    dt_pipeline,
    param_grid=dt_param_grid,
    cv=cv_strategy,
    scoring='f1_macro',
    n_jobs=-1
)
dt_grid.fit(X_train, y_train)

print(f"✅ Hyperparameter Decision Tree Terbaik: {dt_grid.best_params_}")
print(f"Skor Macro F1 Validasi Silang (Train CV): {dt_grid.best_score_:.4f}")

In [ ]:
# Analisis Feature Importance pada Decision Tree
best_dt_model = dt_grid.best_estimator_.named_steps['clf']

# Mendapatkan nama fitur setelah pra-pemrosesan
encoded_cat_names = dt_grid.best_estimator_.named_steps['prep'].named_transformers_['cat'].get_feature_names_out(categorical_features)
all_feature_names = numeric_features + list(encoded_cat_names)

importances = best_dt_model.feature_importances_
feat_imp_df = pd.DataFrame({
    'Fitur': all_feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
top_feats = feat_imp_df.head(10).sort_values('Importance', ascending=True)
plt.barh(top_feats['Fitur'], top_feats['Importance'], color='coral')
plt.title('Top 10 Feature Importances pada Model Decision Tree')
plt.xlabel('Tingkat Kepentingan Relatif (Gini Importance)')
plt.tight_layout()
plt.show()

print("Top 5 Fitur Paling Berpengaruh:")
display(feat_imp_df.head(5))

In [ ]:
# Visualisasi Pohon Keputusan (Depth Terbatas untuk Kemudahan Interpretasi)
plt.figure(figsize=(18, 8))
plot_tree(
    best_dt_model,
    max_depth=2,
    feature_names=all_feature_names,
    class_names=choices,
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title('Struktur Pohon Keputusan (Visualisasi 2 Level Teratas)', fontsize=14)
plt.show()

In [ ]:
# Pemodelan & Tuning: K-Nearest Neighbors (KNN)
knn_pipeline = Pipeline([
    ('prep', knn_preprocessor),
    ('clf', KNeighborsClassifier())
])

knn_param_grid = {
    'clf__n_neighbors': [3, 5, 7, 9, 11, 15, 21, 25],
    'clf__weights': ['uniform', 'distance'],
    'clf__metric': ['euclidean', 'manhattan']
}

print("⏳ Melatih dan melakukan tuning KNN dengan 5-Fold CV...")
knn_grid = GridSearchCV(
    knn_pipeline,
    param_grid=knn_param_grid,
    cv=cv_strategy,
    scoring='f1_macro',
    n_jobs=-1
)
knn_grid.fit(X_train, y_train)

print(f"✅ Hyperparameter KNN Terbaik: {knn_grid.best_params_}")
print(f"Skor Macro F1 Validasi Silang (Train CV): {knn_grid.best_score_:.4f}")

In [ ]:
# Kurva Sensitivitas Nilai k pada KNN (Elbow Curve)
k_values = [1, 3, 5, 7, 9, 11, 15, 19, 23, 27]
cv_scores = []

for k in k_values:
    pipe = Pipeline([
        ('prep', knn_preprocessor),
        ('clf', KNeighborsClassifier(n_neighbors=k, weights='distance'))
    ])
    res = GridSearchCV(pipe, param_grid={}, cv=cv_strategy, scoring='f1_macro', n_jobs=-1)
    res.fit(X_train, y_train)
    cv_scores.append(res.best_score_)

plt.figure(figsize=(10, 5))
plt.plot(k_values, cv_scores, marker='o', linestyle='-', color='darkblue', linewidth=2)
plt.title('Kurva Sensitivitas Nilai k terhadap Macro F1-Score (Cross Validation)')
plt.xlabel('Nilai k (Jumlah Tetangga)')
plt.ylabel('Macro F1-Score (Train CV)')
plt.xticks(k_values)
plt.grid(True)
plt.show()

---  
## 📊 8. Evaluasi dan Perbandingan Model

**Tujuan:**  
1. Menguji model terbaik *Decision Tree* dan *KNN* pada data uji (*unseen test data*).
2. Menghitung dan membandingkan metrik **Accuracy**, **Macro Precision**, **Macro Recall**, dan **Macro F1-Score**.
3. Memvisualisasikan *Confusion Matrix* berdampingan untuk melihat pola kesalahan prediksi antar-kelas.
4. Membahas mengapa metrik **Macro F1-Score** sangat esensial ketika ukuran kelas tidak sepenuhnya seimbang.

In [ ]:
# Model terbaik hasil grid search
best_dt = dt_grid.best_estimator_
best_knn = knn_grid.best_estimator_

# Prediksi pada data uji (X_test)
y_pred_dt = best_dt.predict(X_test)
y_pred_knn = best_knn.predict(X_test)

# Perhitungan metrik evaluasi
target_classes = ['Rendah', 'Sedang', 'Tinggi']

metrics_comparison = pd.DataFrame({
    'Metrik Evaluasi': ['Accuracy', 'Macro Precision', 'Macro Recall', 'Macro F1-Score'],
    'Decision Tree': [
        accuracy_score(y_test, y_pred_dt),
        precision_score(y_test, y_pred_dt, average='macro', zero_division=0),
        recall_score(y_test, y_pred_dt, average='macro', zero_division=0),
        f1_score(y_test, y_pred_dt, average='macro', zero_division=0)
    ],
    'K-Nearest Neighbors (KNN)': [
        accuracy_score(y_test, y_pred_knn),
        precision_score(y_test, y_pred_knn, average='macro', zero_division=0),
        recall_score(y_test, y_pred_knn, average='macro', zero_division=0),
        f1_score(y_test, y_pred_knn, average='macro', zero_division=0)
    ]
})

metrics_comparison['Decision Tree'] = metrics_comparison['Decision Tree'].apply(lambda x: f"{x:.4f} ({x*100:.2f}%)")
metrics_comparison['K-Nearest Neighbors (KNN)'] = metrics_comparison['K-Nearest Neighbors (KNN)'].apply(lambda x: f"{x:.4f} ({x*100:.2f}%)")

print("=== TABEL PERBANDINGAN PERFORMA MODEL PADA DATA UJI ===")
display(metrics_comparison)

In [ ]:
# Visualisasi Confusion Matrix berdampingan
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm_dt = confusion_matrix(y_test, y_pred_dt, labels=target_classes)
disp_dt = ConfusionMatrixDisplay(confusion_matrix=cm_dt, display_labels=target_classes)
disp_dt.plot(cmap='Blues', ax=axes[0], values_format='d')
axes[0].set_title('Confusion Matrix: Decision Tree', fontsize=13)
axes[0].grid(False)

cm_knn = confusion_matrix(y_test, y_pred_knn, labels=target_classes)
disp_knn = ConfusionMatrixDisplay(confusion_matrix=cm_knn, display_labels=target_classes)
disp_knn.plot(cmap='Greens', ax=axes[1], values_format='d')
axes[1].set_title('Confusion Matrix: K-Nearest Neighbors (KNN)', fontsize=13)
axes[1].grid(False)

plt.suptitle('Perbandingan Confusion Matrix pada Data Uji (Test Set)', fontsize=15, y=1.03)
plt.tight_layout()
plt.show()

In [ ]:
# Laporan Klasifikasi Rinci (Classification Report)
print("=" * 60)
print("📋 CLASSIFICATION REPORT: DECISION TREE")
print("=" * 60)
print(classification_report(y_test, y_pred_dt, target_names=target_classes, digits=4, zero_division=0))

print("\n" + "=" * 60)
print("📋 CLASSIFICATION REPORT: K-NEAREST NEIGHBORS (KNN)")
print("=" * 60)
print(classification_report(y_test, y_pred_knn, target_names=target_classes, digits=4, zero_division=0))

### 💡 Mengapa Metrik 'Macro F1-Score' Sangat Penting?
- **Kelemahan Accuracy:** Nilai *Accuracy* hanya mengukur persentase prediksi benar secara agregat. Jika salah satu kelas mendominasi 70% data, model yang hanya menebak kelas mayoritas akan tetap memperoleh akurasi 70%, padahal gagal total mendeteksi kelas lainnya.
- **Keunggulan Macro F1:** *Macro F1-Score* menghitung nilai F1 untuk setiap kelas secara independen lalu mengambil rata-rata aritmetiknya secara setara. Hal ini memberikan **bobot kepentingan yang adil bagi kelas minoritas maupun mayoritas**.

---  
## 💡 9. Interpretasi dan Kesimpulan

Bagian ini merangkum temuan yang dihasilkan secara objektif ke dalam empat pilar utama:

### A. Temuan Deskriptif (Eksplorasi Data)
1. **Pola Pemakaian Smartphone:**  
   - Remaja dalam dataset ini memiliki variasi durasi pemakaian harian yang luas, dengan porsi terbesar dihabiskan untuk media sosial dan hiburan (*gaming*).
2. **Asosiasi dengan Skor Kecanduan:**  
   - Terdapat keterkaitan positif antara total durasi pemakaian harian (`Daily_Usage_Hours`), frekuensi pengecekan ponsel (`Phone_Checks_Per_Day`), serta waktu di media sosial terhadap skor `Addiction_Level`.
   - Sebaliknya, waktu yang dialokasikan untuk kegiatan belajar (`Time_on_Education`) menunjukkan korelasi yang jauh lebih rendah atau bahkan netral/negatif terhadap skor kecanduan.
   - *Catatan:* Keterkaitan ini merupakan **korelasi/asosiasi statistik**, bukan bukti bahwa aplikasi tertentu secara langsung menjadi 'penyebab' timbulnya kecanduan.

### B. Perbandingan Kinerja Model (Decision Tree vs KNN)
1. **Karakteristik Decision Tree:**  
   - Keunggulan utama terletak pada interpretabilitas hierarkisnya (*rule-based*), di mana pemisahan pertama pada pohon biasanya didasarkan pada fitur paling dominan (seperti `Daily_Usage_Hours`).
   - Rentan terhadap perubahan kecil pada batas ambang (*threshold sensitivity*).
2. **Karakteristik KNN:**  
   - Mampu menangkap pola kompleks non-linear berdasarkan kedekatan jarak (*similarity*) antar-profil responden.
   - Sangat bergantung pada kualitas *feature scaling* (StandardScaler) dan sensitif terhadap *curse of dimensionality* jika banyak fitur yang tidak relevan disertakan.
3. **Peringatan Pemilihan Model 'Terbaik':**  
   - Pemilihan model tidak boleh hanya bertumpu pada skor akurasi tertinggi semata. Perhatikan *confusion matrix*: apakah model sering salah memprediksi kategori 'Tinggi' menjadi 'Rendah'? Kesalahan fatal seperti itu memiliki implikasi yang lebih berisiko dalam skenario intervensi konseling.

### C. Keterbatasan Penelitian (*Limitations*)
1. **Sifat Batas Kategori (Heuristik):**  
   - Pengkategorian skor `Addiction_Level` menjadi 3 kelas adalah konstruksi eksploratif peneliti, **bukan batas baku medis/klinis**. Perubahan nilai batas pada variabel `BATAS_KATEGORI` akan secara langsung mengubah label target dan memengaruhi metrik performa model.
2. **Data Cross-Sectional / Observasional:**  
   - Dataset ini mencerminkan potret data satu waktu (*cross-sectional*), sehingga tidak dapat membuktikan hubungan sebab-akibat longitudinal.
3. **Ketiadaan Konteks Kualitatif:**  
   - Motivasi intrinsik remaja dalam menggunakan ponsel (misal stres lingkungan atau kebutuhan bersosialisasi) tidak terekam secara kualitatif mendalam.

### D. Saran & Rekomendasi
1. **Bagi Institusi Pendidikan/Konselor:**  
   - Model ini dapat digunakan sebagai alat penapisan awal (*screening aid*) untuk mengenali profil siswa yang membutuhkan pendampingan manajemen waktu digital, namun setiap tindakan wajib divalidasi dengan asesmen konseling langsung.
2. **Bagi Pengembangan Model Selanjutnya:**  
   - Melakukan pengujian batas kategori berbasis konsensus ahli psikologi anak/remaja.
   - Mencoba algoritma *ensemble* seperti *Random Forest* atau *Gradient Boosting* untuk mengevaluasi apakah performa klasifikasi dapat ditingkatkan lebih lanjut.

---  
## 📚 10. Referensi / Sumber Data

Silakan lengkapi bagian ini dengan sitasi, tautan sumber dataset, atau literatur yang digunakan kelompok Anda:

1. **Sumber Dataset:**  
   - File: `teen_phone_addiction_dataset.csv` (Dataset Asli) & `teen_phone_addiction_prepared.csv` (Dataset Olah).
   - Tautan/Repositori: *[Tuliskan URL repositori dataset atau sumber Kaggle / GitHub jika ada]*

2. **Metodologi & Referensi Pustaka:**  
   - Chapman, P., Clinton, J., Kerber, R., Khabaza, T., Reinartz, T., Shearer, C., & Wirth, R. (2000). *CRISP-DM 1.0: Step-by-step data mining guide*.
   - Pedregosa, F., et al. (2011). *Scikit-learn: Machine Learning in Python*. Journal of Machine Learning Research, 12, 2825-2830.
   - Han, J., Kamber, M., & Pei, J. (2012). *Data Mining: Concepts and Techniques*. Morgan Kaufmann.
   - Provost, F., & Fawcett, T. (2013). *Data Science for Business: What you need to know about data mining and data-analytic thinking*. O'Reilly Media.

3. **Definisi & Literatur Penggunaan Smartphone Remaja:**  
   - *[Tambahkan artikel jurnal atau artikel ilmiah terkait adiksi smartphone pada remaja yang dirujuk oleh kelompok]*